# ROBO CONDUCTOR — 改良版（Colab）

手動で作った3パターンの演奏と、PyBulletで検出した接触に基づく色変化・効果音を生成します。既存のノートブックと元コードは変更しません。実行時に提出用ZIPが必要です。Colabでの実行は未検証です。


In [ ]:
!pip -q install pybullet==3.2.7 numpy imageio imageio-ffmpeg


## プロジェクトをアップロード

`ROBO_CONDUCTOR_submission.zip` を選択します。ファイル名が異なる場合は選択したZIPを使います。信頼できる自分のZIPだけを使用してください。


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
zip_files = [name for name in uploaded if name.endswith(".zip")]
if len(zip_files) != 1:
    raise ValueError("プロジェクトZIPを1つ選択してください")
root = Path("/content/robo_conductor_project").resolve()
root.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_files[0]) as z:
    for info in z.infolist():
        target = (root / info.filename).resolve()
        if not target.is_relative_to(root):
            raise ValueError("安全でないZIP内パスがあります")
    z.extractall(root)
projects = list(root.rglob("conductor.py"))
if len(projects) != 1:
    raise ValueError("conductor.py を1つ含むZIPを指定してください")
project = projects[0].parent
print("プロジェクト:", project)


## 演奏動画を生成

初回は確認用の640×360です。生成には時間がかかります。フルHDで作る場合は `--width 1920 --height 1080` に変更します。


In [ ]:
import subprocess
subprocess.run(["python", "conductor.py", "--output", "output/robo_conductor.mp4", "--log", "output/results.json", "--audio", "output/contacts.wav", "--fps", "30", "--width", "640", "--height", "360"], cwd=project, check=True)


## 結果を確認・保存

成功件数と接触ログを確認してください。接触がない打撃は成功として扱いません。効果音は接触イベントから合成しています。


In [ ]:
import json
from IPython.display import Video, display
result = json.loads((project / "output/results.json").read_text())
print("接触確認:", result["successful_strikes"], "/", result["total_strikes"])
display(Video(str(project / "output/robo_conductor.mp4")))
for name in ["robo_conductor.mp4", "results.json", "contacts.wav"]:
    files.download(str(project / "output" / name))
